# Sweep: scan a closet video on a Colab GPU

Backup for when your Mac is busy or away. Run the cells top to bottom:

1. **Runtime → Change runtime type → T4 GPU** (any GPU works; CPU works too, slowly).
2. Upload the Sweep code (`sweep_code.zip`, made on your Mac with
   `zip -r sweep_code.zip sweep requirements.txt`), *or* set `GIT_URL` if the repo is on GitHub.
3. Optionally upload your current `closet.zip` so new items are **added** to it (duplicates skipped).
4. Upload your closet video (.MOV/.MP4 straight from the iPhone is fine).
5. Download `closet.zip` at the end and unzip it over the `closet/` folder in your Sweep folder on the Mac.

In [ ]:
!nvidia-smi -L || echo "No GPU: scanning will run on CPU (slow). Runtime > Change runtime type > T4 GPU."

## 1. Get the code

In [ ]:
import os, zipfile
from google.colab import files

GIT_URL = ""  # e.g. "https://github.com/you/sweep.git"; leave empty to upload sweep_code.zip

%cd /content
if GIT_URL:
    !rm -rf /content/sweep_repo && git clone -q $GIT_URL /content/sweep_repo
    %cd /content/sweep_repo
else:
    print("Choose sweep_code.zip")
    up = files.upload()
    os.makedirs("/content/sweep_repo", exist_ok=True)
    zipfile.ZipFile(next(iter(up))).extractall("/content/sweep_repo")
    %cd /content/sweep_repo
!ls

## 2. Install dependencies (Colab already has torch with CUDA)

In [ ]:
!pip install -q opencv-python-headless av scikit-learn "transformers>=4.51" pillow
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3. (Optional) Upload your current closet so this scan appends to it
Skip this cell to start a fresh closet. On the Mac: `zip -r closet.zip closet`

In [ ]:
print("Choose closet.zip (or Cancel to start fresh)")
up = files.upload()
if up:
    zipfile.ZipFile(next(iter(up))).extractall("/content/sweep_repo")
    print("existing items:", len(__import__("json").load(open("closet/items.json"))))

## 4. Upload the closet video

In [ ]:
print("Choose your closet video")
up = files.upload()
VIDEO = next(iter(up))
print(VIDEO, round(os.path.getsize(VIDEO) / 1e6, 1), "MB")

## 5. Scan
First run downloads the models (~1.5 GB). Tweak `--threshold` (higher merges more) if items are split or merged wrongly; look at `closet/debug/<scan>/clusters/` to judge.

In [ ]:
!python -m sweep.scan "$VIDEO" --out closet --threshold 0.25

## 6. Look at the results

In [ ]:
import json, math
import matplotlib.pyplot as plt
from PIL import Image

items = json.load(open("closet/items.json"))
cols = 6
rows = math.ceil(len(items) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.6))
for ax in axes.flat:
    ax.axis("off")
for ax, it in zip(axes.flat, items):
    ax.imshow(Image.open(os.path.join("closet", it["thumb"])))
    ax.set_title(f"{it['color']['name']}\n{it['subtype']}", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Download
Unzip over your Sweep folder on the Mac (`unzip -o closet.zip`), replacing `closet/`. Debug images are left out to keep the zip small; set `INCLUDE_DEBUG = True` to keep them.

In [ ]:
INCLUDE_DEBUG = False
!rm -f closet.zip
if INCLUDE_DEBUG:
    !zip -qr closet.zip closet -x "closet/uploads/*"
else:
    !zip -qr closet.zip closet -x "closet/uploads/*" "closet/debug/*"
!ls -lh closet.zip
files.download("closet.zip")